# Heatwave Analysis: Table
State & Municipalities — Unified Master Table for All Regions

**Final Project:** Heatwave Analysis in Rio de Janeiro State

**Course:** FA25-BL-EAS-G690-29302

**Professor:** Travis Allen O'Brien

**Step:** Heatwave Calculations

**Author:** Rafaela Quintella Veiga

**Date:** December 2025

---

## Overview
This notebook consolidates the detection and characterization of heatwaves across two spatial scales:

1. Regional Scale (State Level):
   Using the spatially aggregated time series for the entire state of Rio de Janeiro.

2. Local Scale (Municipal Level):
   Applying the heatwave detection algorithm individually to each of the 92 municipalities.

The final output is a single, unified master table containing summary metrics and a complete list of heatwave events for all regions.

### Methodology
The analysis is based on the custom algorithm implemented in core_heatwave.py.

Key components include:

- Climatological Baseline: Percentile thresholds (80th, 90th, 95th) computed using a moving window over the baseline period (1981–2010).

- Event Detection: Heatwaves defined as ≥3 consecutive days above the percentile threshold.

- Event Characterization:
   - Duration category
   - Mean and maximum intensity
   - Maximum temperature
   - Seasonal classification 
   - Annual frequency

Both state-level and municipal-level datasets are processed using the same methodology to maintain consistency.

### Output
This notebook produces two master CSV files:

1. MASTER_metrics_all_regions.csv
Contains summary statistics for each region and percentile. 

2. MASTER_events_all_regions.csv
Contains the complete event catalog for all state and municipal regions.

These datasets serve as the foundation for all subsequent spatial analyses, visualizations, and climatological comparisons.

In [ ]:
# Install dependencies (silently) if not present
try:
    import tqdm
except ImportError:
    !pip install tqdm -q

In [1]:
import pandas as pd
import os
import sys
from tqdm import tqdm

# --- Import Custom Library ---
# Ensures Python finds 'core_heatwave.py' in the current directory
sys.path.append(os.getcwd())
import core_heatwave as hw

print("✅ Libraries imported and environment set up.")

✅ Libraries imported and environment set up.


## 1. Data Configuration

In this section, we define the input file paths for the temperature time series and specify where the consolidated outputs will be saved.  
We also set the global analysis parameters used by the heatwave detection algorithm.

- **Reference Period (Baseline):** 1981–2010.
- **Percentiles Analyzed:** 80th, 90th, and 95th.
- **Minimum Event Duration:** 3 consecutive days above the threshold.

The paths below should be adapted to your local environment if the directory structure differs.


In [2]:
# --- Input File Paths ---
# Adjust these paths according to your local machine
PATH_STATE = r"C:\Users\rafaq\EASG690\Final_Project\scripts\\heatwave_analysis\processed_data\results_rj_state.csv"
PATH_MUNI = r"C:\Users\rafaq\EASG690\Final_Project\scripts\heatwave_analysis\processed_data\results_rj_municipalities.csv"

# --- Output Directory ---
OUTPUT_DIR = r"C:\Users\rafaq\EASG690\Final_Project\scripts\heatwave_analysis\outputs"

# --- Analysis Parameters ---
REF_PERIOD = (1981, 2010)
PERCENTILES = [80, 90, 95]
MIN_DURATION = 3

# Create output directory if it doesn't exist
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    print(f"📂 Directory created: {OUTPUT_DIR}")
else:
    print(f"📂 Output directory set: {OUTPUT_DIR}")

# Accumulator lists for final results
all_metrics = []
all_events = []

📂 Output directory set: C:\Users\rafaq\EASG690\Final_Project\scripts\heatwave_analysis\outputs


---
## 2. Processing: State Level (Regional)

In this section, we load the time series corresponding to the **spatially aggregated temperature** for the entire state of Rio de Janeiro.  
This regional series serves as a **baseline benchmark** for comparing municipal-level heatwave behavior.

**Objective:**  
Apply the `HeatwaveDetector` using the climatological baseline (1981–2010) to compute the occurrence, duration, and intensity of heatwaves at the **state scale**.

**Workflow:**
1. Load the processed statewide CSV (`results_rj_state.csv`).
2. Initialize the detector with the temperature variable (`t2m`) and reference period.
3. Run the analysis for the 80th, 90th, and 95th percentile thresholds.
4. Store the resulting metrics and event catalog for later consolidation.


In [3]:
print("🏛️ Starting STATE processing...")

if os.path.exists(PATH_STATE):
    try:
        # 1. Load Data
        df_state = pd.read_csv(PATH_STATE)
        
        # 2. Initialize Detector (Custom Library)
        detector_state = hw.HeatwaveDetector(
            df=df_state, 
            variable='t2m', 
            date_col='time', 
            reference_period=REF_PERIOD
        )
        
        # 3. Run Analysis
        events_state, metrics_state = detector_state.analyze(
            percentiles=PERCENTILES, 
            min_duration=MIN_DURATION
        )
        
        # 4. Consolidate
        if not metrics_state.empty:
            # Explicitly label as 'RJ_State' to distinguish from cities later
            metrics_state['region'] = 'RJ_State'
            events_state['region'] = 'RJ_State'
            
            all_metrics.append(metrics_state)
            all_events.append(events_state)
            print("✅ RJ State processed successfully!")
        else:
            print("⚠️ No heatwaves detected for the State.")
            
    except Exception as e:
        print(f"❌ Critical error processing State: {e}")
else:
    print(f"⚠️ State file not found: {PATH_STATE}")

🏛️ Starting STATE processing...
✅ RJ State processed successfully!


---
## 3. Processing: Municipal Level (Local)

In this step, we extend the heatwave analysis to all **92 municipalities** in the state of Rio de Janeiro.

Processing each municipality individually allows us to:
- Capture **local-scale variability**,  
- Identify **regional contrasts**, and  
- Compare municipal sensitivity against the statewide benchmark.

A progress bar (`tqdm`) is used to track execution, as running the algorithm across ~92 locations and multiple percentiles requires substantial computation.

**Workflow:**
1. Load the full municipal CSV (`results_rj_municipalities.csv`).
2. Extract the list of unique municipality names.
3. Loop through each municipality:
   - Subset the data.
   - Initialize the detector.
   - Run the heatwave analysis.
   - Store both metrics and event-level results.
4. Append all outputs to accumulator lists for later consolidation.


In [4]:
print("🏙️ Starting MUNICIPALITIES processing...")

if os.path.exists(PATH_MUNI):
    df_all_muni = pd.read_csv(PATH_MUNI)
    
    # Identify unique cities/regions in the file
    if 'region' in df_all_muni.columns:
        cities = df_all_muni['region'].unique()
    else:
        cities = df_all_muni['NM_MUN'].unique() # Fallback if column name changes
    
    # Loop with progress bar
    for city in tqdm(cities, desc="Processing Cities"):
        try:
            # 1. Filter current city
            df_city = df_all_muni[df_all_muni['region'] == city].copy()
            
            # 2. Initialize Detector
            detector_city = hw.HeatwaveDetector(
                df=df_city, 
                variable='t2m', 
                date_col='time', 
                reference_period=REF_PERIOD
            )
            
            # 3. Analysis
            events, metrics = detector_city.analyze(
                percentiles=PERCENTILES, 
                min_duration=MIN_DURATION
            )
            
            # 4. Append Results
            if not metrics.empty:
                metrics['region'] = city
                events['region'] = city
                
                all_metrics.append(metrics)
                all_events.append(events)
                
        except Exception as e:
            # Silent error logging to avoid stopping the loop
            # print(f"Error in {city}: {e}")
            pass
            
    print(f"✅ Municipal processing finished. Total regions analyzed: {len(cities)}")

else:
    print(f"⚠️ Municipalities file not found: {PATH_MUNI}")

🏙️ Starting MUNICIPALITIES processing...


Processing Cities: 100%|██████████| 92/92 [03:00<00:00,  1.96s/it]

✅ Municipal processing finished. Total regions analyzed: 92


---
## 4. Consolidation and Export

At this stage, all outputs from the State and Municipal analyses are merged into two unified master tables. These datasets serve as the foundation for mapping, statistical modeling, and climatological comparisons across all regions.

Generated Outputs

1. MASTER_metrics_all_regions.csv
A regional summary table where each row corresponds to a unique combination of

    . Region (e.g., RJ_State, Petrópolis, Rio de Janeiro, etc.)

    . Percentile threshold analyzed (80th, 90th, 95th)

    . Duration category (3–4 days, 5–7 days, >7 days)

For each region × percentile × duration class, the table contains

    . total_events – Total number of detected heatwaves

    . avg_duration – Mean duration (days)

    . avg_intensity – Mean intensity anomaly

    . max_intensity – Maximum intensity anomaly

    . annual_frequency – Number of heatwaves per year (normalized by record length)

**Use:** Regional summaries, intensity/frequency comparisons, and map-ready climatological metrics.

2. MASTER_events_all_regions.csv
A complete catalog of all heatwave events detected across the entire state and all municipalities.

Each row corresponds to a single event, containing

Event metadata

        . region

        . percentile

        . duration_category

Temporal structure

        . start_date

        . end_date

        . duration_days

Event thermodynamic characteristics

        . mean_temperature

        . max_temperature

        . intensity_mean

        . intensity_max

Seasonal and annual attribution

        . year

        . season (DJF, MAM, JJA, SON)

**Use: Temporal analyses, trend assessments, event chronology, and all map-based frequency calculations.**

**Purpose of Consolidation**

Together, these two master datasets

Provide a standardized, multi-scale heatwave database for the entire state.

Enable analyses such as

    . A standardized, multi-scale heatwave database for all regions

    . Inputs for spatial pattern analyses and identification of highly affected municipalities

    . Support for seasonal and interannual variability assessments

    . Event-level comparisons across the state

In [5]:
print("💾 Saving MASTER consolidated files...")

if all_metrics:
    # 1. Concatenation
    final_metrics_df = pd.concat(all_metrics, ignore_index=True)
    final_events_df = pd.concat(all_events, ignore_index=True)
    
    # 2. Column Reordering (Aesthetics)
    # Ensure 'region' is the first column
    cols_metrics = ['region'] + [c for c in final_metrics_df.columns if c != 'region']
    final_metrics_df = final_metrics_df[cols_metrics]
    
    cols_events = ['region'] + [c for c in final_events_df.columns if c != 'region']
    final_events_df = final_events_df[cols_events]
    
    # 3. Export to CSV
    path_metrics = os.path.join(OUTPUT_DIR, "MASTER_metrics_all_regions.csv")
    path_events = os.path.join(OUTPUT_DIR, "MASTER_events_all_regions.csv")
    
    final_metrics_df.to_csv(path_metrics, index=False)
    final_events_df.to_csv(path_events, index=False)
    
    print(f"✅ Files successfully generated in: {OUTPUT_DIR}")
    print(f"   - {os.path.basename(path_metrics)} (Statistical Summary)")
    print(f"   - {os.path.basename(path_events)} (Detailed Events List)")
    
    # Preview
    print("\n🔍 Data Preview (Notice 'RJ_State' mixed with cities):")
    display(final_metrics_df.head())
    
    # Optional: Verify State data exists
    if 'RJ_State' in final_metrics_df['region'].values:
         display(final_metrics_df[final_metrics_df['region'] == 'RJ_State'].head(2))

else:
    print("❌ No data was processed.")

💾 Saving MASTER consolidated files...
✅ Files successfully generated in: C:\Users\rafaq\EASG690\Final_Project\scripts\heatwave_analysis\outputs
   - MASTER_metrics_all_regions.csv (Statistical Summary)
   - MASTER_events_all_regions.csv (Detailed Events List)

🔍 Data Preview (Notice 'RJ_State' mixed with cities):


,region,percentile,duration_category,total_events,avg_duration,avg_intensity,max_intensity,annual_frequency
0,RJ_State,80,3-4 days,389,3.331620,1.227999,6.528318,4.576471
1,RJ_State,80,5-7 days,161,5.515528,1.468757,6.953128,1.894118
2,RJ_State,80,>7 days,60,10.700000,1.666138,7.088921,0.705882
3,RJ_State,90,3-4 days,191,3.308901,1.058966,5.472226,2.247059
4,RJ_State,90,5-7 days,54,5.685185,1.256536,5.515536,0.635294


,region,percentile,duration_category,total_events,avg_duration,avg_intensity,max_intensity,annual_frequency
0,RJ_State,80,3-4 days,389,3.331620,1.227999,6.528318,4.576471
1,RJ_State,80,5-7 days,161,5.515528,1.468757,6.953128,1.894118
